# nano-dsv4.1f — Kaggle TPU v5e-8 native smoke

This notebook exercises the **v5e-native training backend**: TPU Splash for CSA2 attention, static-capacity expert parallelism for the 8 routed experts, and block rematerialization. It keeps the 32K vocabulary and starts at `T=1024` to validate compilation/HBM before real data.


## Repository + TPU runtime bootstrap — before JAX starts

Clone/install before importing JAX. Kaggle can pair a recent Python `jax` package with an older TPU runtime, so this cell also installs `jax[tpu]` at the **already-installed JAX version** to align `jaxlib` + `libtpu` for Pallas/Splash. `src/` is inserted directly into `sys.path` so the fresh checkout is visible without a kernel restart.


In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import os, subprocess, sys

REPO_URL = "https://github.com/xiayicheng3-code/nano-dsv4.1f.git"
REPO_REF = os.environ.get("NANO_DSV41F_REF", "main")
TARGET = Path("/kaggle/working/nano-dsv4.1f")

def is_repo_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "nano_dsv41f").exists()

candidates = [Path.cwd(), Path.cwd().parent, TARGET]
root = next((p.resolve() for p in candidates if is_repo_root(p)), None)

if root is None:
    if TARGET.exists():
        raise RuntimeError(f"{TARGET} exists but is not a valid checkout; remove/rename it and rerun.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(TARGET)], check=True)
    root = TARGET
else:
    raw_status = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=root, text=True
    ).splitlines()
    generated = ("nano_dsv41f.egg-info/", "__pycache__/", ".ipynb_checkpoints/")
    dirty = [line for line in raw_status if not any(token in line for token in generated)]
    if dirty:
        raise RuntimeError(
            "Existing checkout has local changes; refusing to overwrite:\n" + "\n".join(dirty)
        )

subprocess.run(["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=root, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=root, check=True)
os.chdir(root)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
    check=True,
)

# Align the TPU runtime before importing JAX. This keeps the Python JAX version unchanged
# while installing the matching jaxlib/libtpu set required by Pallas/Splash.
jax_pkg_version = version("jax")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "--upgrade-strategy", "only-if-needed", f"jax[tpu]=={jax_pkg_version}",
    ],
    check=True,
)

def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"

print("aligned TPU packages:", {
    "jax": package_version("jax"),
    "jaxlib": package_version("jaxlib"),
    "libtpu": package_version("libtpu"),
})

src_path = str((root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
print("repo root:", root)
print("requested ref:", REPO_REF)
print("commit:", commit)
print("src import path:", src_path)


## TPU runtime preflight


In [ ]:
import jax
import jax.numpy as jnp
from jax.extend import backend as jax_backend

print("JAX:", jax.__version__)
print("backend:", jax.default_backend())
print("platform_version:", jax_backend.get_backend().platform_version)
print("devices:", jax.devices())
print("process_count:", jax.process_count(), "local_device_count:", jax.local_device_count())

if not jax.devices() or jax.devices()[0].platform != "tpu":
    raise RuntimeError("Select TPU in Kaggle Settings > Accelerator, restart the session, then rerun.")


## v5e topology + native backend


In [ ]:
from nano_dsv41f import (
    ModelConfig, TrainConfig, TPUNativeConfig, V5E,
    make_v5e_mesh, runtime_report, semantic_axes,
    validate_sequence_length, validate_v5e_runtime,
)

config = ModelConfig()
train_config = TrainConfig(seq_len=4096)
native_config = TPUNativeConfig(moe_capacity_factor=2.0)

print(runtime_report())
for warning in validate_v5e_runtime():
    print("WARNING:", warning)

mesh = make_v5e_mesh()
print("mesh:", mesh)
print("v5e:", V5E)
print("native backend:", native_config)
print("vocab_size:", config.vocab_size)
print("parameter/batch axes:", semantic_axes(config, mesh))
for warning in validate_sequence_length(train_config.seq_len, config, mesh):
    print("WARNING:", warning)


The outer model keeps the topology-aware Auto mesh. Inside hot operations, `shard_map` uses a flat 8-chip `tp` view: **Q rows are sequence-sharded for Splash, and each of the 8 routed experts is resident on one chip.** Expert matrices are never gathered per token.


## Direct-to-shard BF16 initialization


In [ ]:
from nano_dsv41f import (
    init_model_sharded_mixed_precision, init_optimizer_state_sharded,
    memory_report, precision_summary,
)

params, param_specs, param_shardings = init_model_sharded_mixed_precision(
    jax.random.PRNGKey(0), config, mesh, payload_dtype=jnp.bfloat16
)
jax.block_until_ready(jax.tree_util.tree_leaves(params)[0])
print("dtypes:", precision_summary(params))
print("parameter memory:", memory_report(params, param_specs, mesh))

opt_state, opt_state_shardings = init_optimizer_state_sharded(
    params, param_specs, config, mesh
)
jax.block_until_ready(jax.tree_util.tree_leaves(opt_state)[0])
print("optimizer state initialized")


## Build the native base executable


In [ ]:
from nano_dsv41f import compile_pretrain_step

base_step = compile_pretrain_step(
    params, opt_state, param_specs, config, train_config, mesh,
    include_indexer=False, n_segments=None, native_config=native_config,
)
print("native base_step created")


## Synthetic `T=1024` smoke batch


In [ ]:
import numpy as np
from nano_dsv41f import put_training_batch

smoke_t = 1024
host_ids = np.arange(smoke_t, dtype=np.int32)[None, :] % config.vocab_size
host_segments = np.zeros_like(host_ids, dtype=np.int32)
host_mask = np.ones_like(host_ids, dtype=bool)

ids, segments, token_mask = put_training_batch(
    host_ids, host_segments, host_mask, config, mesh
)
print("input sharding:", ids.sharding)
print("local shard:", ids.addressable_shards[0].data.shape)


## Compile diagnostics

This is the main gate. The earlier dense-reference run died from memory pressure; the first native run reached Splash lowering but exposed a stale Kaggle `libtpu`. If this cell succeeds, record compiler memory and collective counts before executing a training step.


In [ ]:
from nano_dsv41f import compile_diagnostics

zero_step = jnp.asarray(0, jnp.int32)
compiled_base, diag = compile_diagnostics(
    base_step, params, opt_state, ids, segments, zero_step, token_mask
)
print("collectives:", diag["collectives"])
print("compiler memory:", diag["memory"])
print({
    k: v for k, v in diag["cost"].items()
    if any(t in k.lower() for t in ("flop", "byte", "transcend"))
})


## First native training step


In [ ]:
params, opt_state, metrics = compiled_base(
    params, opt_state, ids, segments, zero_step, token_mask
)
jax.block_until_ready(metrics["loss"])
print({
    k: float(v)
    for k, v in metrics.items()
    if getattr(v, "ndim", 1) == 0
})


## Late-indexer executable — only after base succeeds

The late phase requests a stop-gradient Splash LSE teacher pass for selected-query distillation. Build it only after the base path has proven its memory behavior.


In [ ]:
late_indexer_step = compile_pretrain_step(
    params, opt_state, param_specs, config, train_config, mesh,
    include_indexer=True, n_segments=1, native_config=native_config,
)
print("late_indexer_step wrapper created; compile/profile it only after the base smoke is healthy")


## Next

If compiler memory is comfortable and the first loss is finite, move to `T=4096`, then attach the pretrained tokenizer + packed dataset. Keep the commit SHA, JAX/jaxlib/libtpu versions, TPU platform version, sequence length, compiler memory, collective counts, and step time for every benchmark.
